# Notebook 03: Huấn luyện và Phân tích Hệ số Hồi quy Logistic (Logistic Regression)

**Người phụ trách chính:** TV2 (Decision Tree & Logistic Regression).

### Mục tiêu nghiên cứu:
1. Đánh giá vai trò của chuẩn hóa `StandardScaler` lên các biến số (`tenure`, `MonthlyCharges`, `TotalCharges`).
2. Tìm kiếm siêu tham số điều chuẩn `C` và cơ chế trọng số lớp `class_weight='balanced'` để xử lý mất cân bằng dữ liệu.
3. Rút trích và phân tích hệ số hồi quy $\beta$ (Log-Odds) và Odds Ratio $\exp(\beta)$ nhằm xác định các yếu tố thúc đẩy churn và giữ chân khách hàng.


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, os.path.abspath(os.path.join('..', 'src')))
from telco_churn.data import load_raw_data
from telco_churn.schema import validate_records, ALL_FEATURE_COLUMNS, TARGET_COLUMN
from telco_churn.training import run_grid_search
from telco_churn.evaluation import plot_logistic_regression_coefficients

sns.set_theme(style='whitegrid')


## 1. Nạp dữ liệu và Chạy 5-Fold GridSearch cho Logistic Regression


In [ ]:
df_raw = load_raw_data('../telco-customer-churn-on-icp4d/data/Telco-Customer-Churn.csv')
valid_df, _ = validate_records(df_raw, mode='training')
manifest = pd.read_csv('../data/splits/split_manifest.csv')
dev_manifest = manifest[manifest['split'] == 'dev']
dev_df = valid_df.merge(dev_manifest[['customerID', 'fold']], on='customerID')
X_dev = dev_df[ALL_FEATURE_COLUMNS].copy()
y_dev = dev_df[TARGET_COLUMN].copy()
folds = dev_df['fold'].values

lr_res = run_grid_search('logistic_regression', X_dev, y_dev, folds)
print('Cấu hình tối ưu:', lr_res['best_params'])
print(f'Mean CV F1: {lr_res["mean_f1"]:.4f} ± {lr_res["std_f1"]:.4f}')
print(f'Mean CV Recall: {lr_res["mean_recall"]:.4f} | ROC-AUC: {lr_res["mean_roc_auc"]:.4f}')


## 2. Trực quan hóa Hệ số Log-Odds (Regression Coefficients)


In [ ]:
fig = plot_logistic_regression_coefficients(lr_res['fitted_pipeline'], top_n=14)
plt.show()


## 3. Phân tích Chi tiết Tác động của các Yếu tố (Odds Ratios)
Bảng tổng hợp các hệ số hồi quy quan trọng nhất và tỷ lệ chênh lệch xác suất (Odds Ratio):


In [ ]:
pipe = lr_res['fitted_pipeline']
classifier = pipe.named_steps['classifier']
prep = pipe.named_steps['preprocessor']
feature_names = prep.get_feature_names_out()
coefs = classifier.coef_[0]

df_coef = pd.DataFrame({
    'Feature': [f.replace('num__', '').replace('cat__', '') for f in feature_names],
    'Coefficient (Beta)': coefs,
    'Odds Ratio (exp(Beta))': np.exp(coefs)
}).sort_values(by='Coefficient (Beta)', ascending=False)

print('=== TOP ĐẶC TRƯNG LÀM TĂNG NGUY CƠ RỜI BỎ (CHURN DRIVERS) ===')
print(df_coef.head(6).to_string(index=False))

print('\n=== TOP ĐẶC TRƯNG GIỮ CHÂN KHÁCH HÀNG (RETENTION DRIVERS) ===')
print(df_coef.tail(6).to_string(index=False))


## 4. Kết luận Nghiệp vụ từ Mô hình Hồi quy Logistic
1. **Nhóm tăng nguy cơ rời bỏ:** Hợp đồng tháng (`Contract_Month-to-month`), mạng cáp quang (`InternetService_Fiber optic`), hình thức thanh toán séc điện tử (`PaymentMethod_Electronic check`).
2. **Nhóm bảo vệ và gắn bó:** Thời gian gắn bó lâu năm (`tenure`), hợp đồng 2 năm (`Contract_Two year`), và có đăng ký các dịch vụ bổ trợ an ninh/hỗ trợ (`OnlineSecurity`, `TechSupport`).
3. **Cảnh báo diễn giải:** Hệ số phản ánh mối liên hệ thống kê (statistical association) sau khi đã kiểm soát các biến khác, không tự ý suy diễn thành tác động nhân quả thuần túy (causality).
